In [99]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import re

In [100]:
time_df = pd.read_csv("testbench_results.csv")
size_df = pd.read_csv("testbench_size_results.csv")

In [101]:
keys = [
    "FILE_NAME",
    "MAX_HEIGHT",
    "PHRASE_NUM"]

size_df_reduced = (size_df.groupby(keys, as_index=False)["BYTES"].mean())

merged_df = time_df.merge(
    size_df_reduced,
    on=keys,
    how="left"
)

merged_df["BYTES"].isna().sum()
check = (
    size_df
    .groupby(keys)["BYTES"]
    .nunique()
)

print(check[check > 1])


Series([], Name: BYTES, dtype: int64)


In [102]:
df = merged_df

# Extract algorithm from file path
df["PATH"] = df["FILE_NAME"].astype(str)
df["ALGORITHM"] = df["PATH"].apply(
    lambda p: os.path.normpath(p).split(os.sep)[-2]
)

df["FILENAME"] = df["PATH"].apply(
    lambda p: os.path.splitext(os.path.basename(p))[0]
)


# Extract dataset from file path
def extract_dataset(name):
    if "_h" in name:
        return name.split("_h")[0].split("/")[-1].split(".")[0]
    if "_lz-" in name:
        return name.split("_lz-")[0].split("/")[-1].split(".")[0]
    return name.split("/")[-1].split(".")[0]

# /home/timmo/Code/lzhb-testbench/res/c4/influenza_h5_a0.1_b0.1_g0_c4.lzcp
df["ALPHA"] = df["FILE_NAME"].apply(lambda n: re.search(r"_a([\d.]+)", n).group(1) if re.search(r"_a([\d.]+)", n) else None)
df["BETA"] = df["FILE_NAME"].apply(lambda n: re.search(r"_b([\d.]+)", n).group(1) if re.search(r"_b([\d.]+)", n) else None)

df["DATASET"] = df["FILE_NAME"].apply(extract_dataset)


def extract_height(name):
    m = re.search(r"_h(\d+)", name)
    return int(m.group(1)) if m else None

df["HEIGHT_BOUND"] = df["FILE_NAME"].apply(extract_height)
df["IS_HEIGHT_BOUND"] = df["HEIGHT_BOUND"].notna()

df[["FILE_NAME", "DATASET"]].head(10)




,FILE_NAME,DATASET
0,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
1,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
2,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
3,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
4,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
5,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
6,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
7,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
8,/home/timmo/Code/lzhb-testbench/res/c1/einstei...,einstein
9,/home/timmo/Code/lzhb-testbench/res/c1/einstei...,einstein


In [103]:
extract_dataset("/home/timmo/Code/lzhb-testbench/res/kpp3/coreutils.lzcp")
#extract_dataset("/home/timmo/Code/lzhb-testbench/res/lzlmocc/coreutils_lz-lmocc.lzcp")
df[df["ALGORITHM"] == "kpp3"]

,TIMESTAMP,FILE_NAME,REPEATS,BATCH_SIZE,PHRASE_NUM,MAX_HEIGHT,AVG_HEIGHT,VAR_HEIGHT,MAX_LENGTH,AVG_LENGTH,...,AVERAGE_STRING_CONSEC_CHAR_TIME_NS,BYTES,PATH,ALGORITHM,FILENAME,ALPHA,BETA,DATASET,HEIGHT_BOUND,IS_HEIGHT_BOUND


In [104]:
DATASET = "coreutils"

huffman_values = {"influenza" : 42.473017, "coreutils" : 141.320725, "einstein" : 291.834463}

df_ds = df[df["DATASET"] == DATASET]
df_flat = df_ds[~df_ds["IS_HEIGHT_BOUND"]]
df_flat[["ALGORITHM", "BYTES", "AVERAGE_ACCESS_CHAR_TIME"]]

df_not_flat = df_ds[df_ds["IS_HEIGHT_BOUND"]]

In [105]:
def summarize_height_group(df_group, space_percentile=0.1):
    """
    df_group: subset of df_not_flat for a particular algorithm
    space_percentile: fraction (0.1 = 10%) of minimum space to filter
    Returns: one row (lowest avg time within small-space region)
    """
    min_space = df_group["BYTES"].min()
    space_threshold = min_space * (1 + space_percentile)

    small_space_df = df_group[df_group["BYTES"] <= space_threshold]

    idx_min_time_small_space = small_space_df["AVERAGE_ACCESS_CHAR_TIME"].idxmin()
    row_min_time_small_space = df_group.loc[idx_min_time_small_space]

    return row_min_time_small_space.to_frame().T

  
  

df_hb = df_not_flat.copy()
# Group by algorithm and dataset (or category if you have one)
summary_hb = df_hb.groupby("ALGORITHM").apply(lambda g: summarize_height_group(g)).reset_index(drop=False)

# Error in benchmark: We divided by batchsize twice
df_flat["AVERAGE_ACCESS_CHAR_TIME"] = (
    df_flat["AVERAGE_ACCESS_CHAR_TIME"].astype(float)
    * df_flat["BATCH_SIZE"].astype(float)
).fillna(0)  # or fill with original value

summary_hb["AVERAGE_ACCESS_CHAR_TIME"] = (
    summary_hb["AVERAGE_ACCESS_CHAR_TIME"].astype(float)
    * summary_hb["BATCH_SIZE"].astype(float)
).fillna(0)

# Error in consecutive benchmark: We divided by repeats twice
df_flat["AVERAGE_CONSEC_CHAR_TIME_NS"] = (
    df_flat["AVERAGE_CONSEC_CHAR_TIME_NS"].astype(float)
    * df_flat["REPEATS"].astype(float)
).fillna(0)  # or fill with original value

summary_hb["AVERAGE_CONSEC_CHAR_TIME_NS"] = (
    summary_hb["AVERAGE_CONSEC_CHAR_TIME_NS"].astype(float)
    * summary_hb["REPEATS"].astype(float)
).fillna(0)

df_flat["KB"] = df_flat["BYTES"] / 1024
summary_hb["KB"] = summary_hb["BYTES"] / 1024

df_flat["MB"] = df_flat["BYTES"] / (1024 * 1024)
summary_hb["MB"] = summary_hb["BYTES"] / (1024 * 1024)

In [106]:
df_other = pd.read_csv("gracli_results.csv")
#df_other = pd.read_csv("gracli_subseq_results.csv")
# Keep only the dataset you are plotting
df_other = df_other[df_other["DATASET"] == DATASET].copy()

# Normalize column names / semantics
df_other["BYTES"] = df_other["malloc_count_after_construction_bytes"]
df_other["MB"] = df_other["BYTES"] / (1024 * 1024)

df_other["AVERAGE_ACCESS_CHAR_TIME"] = (
    df_other["QUERY_TIME_TOTAL"] / (df_other["QUERIES"])
)

# They are in milliseconds instead of nanoseconds. Convert to ns
df_other["AVERAGE_ACCESS_CHAR_TIME"] = df_other["AVERAGE_ACCESS_CHAR_TIME"] * 1e6  # ms to ns
df_other["ALGORITHM"] = "Other: " + df_other["TYPE"]

df_other["TYPE"] = "other"
df_other["MARKER"] = "diamond"   # visually distinct
df_other["CATEGORY"] = "other-algorithms"

df_other.head()

,DATASET,TYPE,INPUT_SIZE,QUERIES,SPACE,CONSTRUCTION_TIME,QUERY_TIME_TOTAL,malloc_count_TOTAL,malloc_count_PEAK,malloc_count_after_construction_bytes,BYTES,MB,AVERAGE_ACCESS_CHAR_TIME,ALGORITHM,MARKER,CATEGORY
2,coreutils,other,205281778,512000,18016040,4354,3543,1734358396,1665717780,18020575,18020575,17.185760,6.919922e+03,Other: lzend,diamond,other-algorithms
5,coreutils,other,205281778,512000,5009776,1664,1813160,343323182,204881275,107453166,107453166,102.475325,3.541328e+06,Other: native_grammar,diamond,other-algorithms
8,coreutils,other,205281778,512000,9821068,69787,1518,3632860899,204881292,112264475,112264475,107.063746,2.964844e+03,Other: sample_scan_512,diamond,other-algorithms


In [107]:
import plotly.express as px
import matplotlib.cm as cm

# --- Assign colors per algorithm ---
all_algorithms = pd.concat([df_flat["ALGORITHM"], summary_hb["ALGORITHM"]]).unique()
cmap = cm.get_cmap("tab10", len(all_algorithms))
def mpl_color_to_rgb_str(color):
    r, g, b, *_ = color
    return f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
algo_color_map = {algo: mpl_color_to_rgb_str(cmap(i)) for i, algo in enumerate(all_algorithms)}

OTHER_COLOR = "rgb(80,80,80)"

for algo in df_other["ALGORITHM"].unique():
    algo_color_map[algo] = OTHER_COLOR


# --- Flat data ---
df_flat_plot = df_flat.copy()
df_flat_plot["TYPE"] = "flat"
df_flat_plot["MARKER"] = "circle"
df_flat_plot["COLOR_ALGO"] = df_flat_plot["ALGORITHM"].map(algo_color_map)

# --- Height-bounded data ---
df_hb_plot = summary_hb.copy()
df_hb_plot["TYPE"] = "height-bounded"
df_hb_plot["MARKER"] = "diamond"
df_hb_plot["COLOR_ALGO"] = df_hb_plot["ALGORITHM"].map(algo_color_map)


df_other_plot = df_other.copy()
df_other_plot["TYPE"] = "other"

df_plot = pd.concat(
    [df_flat_plot, df_hb_plot, df_other_plot],
    ignore_index=True
)



# --- Plot ---
fig = px.scatter(
    df_plot,
    x="MB",
    y="AVERAGE_ACCESS_CHAR_TIME",
    color="ALGORITHM",          
    symbol="TYPE",              
    hover_data=[
        "ALGORITHM", "TYPE", "MB",
        "AVERAGE_ACCESS_CHAR_TIME",
        "HEIGHT_BOUND", "ALPHA", "BETA"
    ],
    color_discrete_map=algo_color_map,
    labels={
        "MB": "Space (MB)",
        "AVERAGE_ACCESS_CHAR_TIME": "Average character access time (ns)",
        "ALGORITHM": "Algorithm",
        "TYPE": "Variant"
    },
    title=f"{DATASET}: Space vs Access Time | 51.200.000 characters"
)

# Add a vertical line for a datapoint without average_access_char_time (e.g., only space known)
# Example: g contains the row, use its MB and ALGORITHM for annotation


fig.add_vline(
    x=huffman_values[DATASET],
    line_dash="dash",
    line_color=algo_color_map.get("Huffman Encoding", "black"),
    annotation_text=f"Huffman Encoding",
    annotation_position="top right",
    annotation_font_color=algo_color_map.get("Huffman Encoding", "black"),
    opacity=0.7
)

fig.update_traces(
    marker=dict(size=12, line=dict(width=1, color="black"))
)

fig.update_layout(legend_title_text='Algorithm')
fig.write_html(f"{DATASET}_plot.html", include_plotlyjs='cdn')
fig.show()



/tmp/ipykernel_13694/2406697887.py:6: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.

